# NB5b (2/3) — foundation-model forecasts (Colab GPU, or locally with `.venv_fm`)

D23, D59. Runs `tools/fm_infer.py` for each setting of the grid and writes `results/nb5b_runs/<model>_c<context>_<covariates>.parquet` with a JSON record (versions, weights revision, input hashes, run time). Resumable: a setting whose output exists for the current inputs is skipped.

**On Colab:** the Drive mirror of the repository must hold `tools/fm_infer.py`, `src/pvcold/` , `requirements-fm.txt` and `results/nb5b_inputs/` (from NB5b_inputs). Choose a GPU runtime, set `DRIVE_REPO`, run all cells, then copy `results/nb5b_runs/` back to the local repository (the master copy). End with `drive.flush_and_unmount()` (last cell) and Ctrl+S.

**Locally:** the separate environment `.venv_fm` (`requirements-fm.txt`) runs on the CPU; slow (Chronos-2 about 0.3 issue times per second under load).

In [ ]:
import os, sys, json, hashlib, subprocess, time
from pathlib import Path
ON_COLAB = 'google.colab' in sys.modules
DRIVE_REPO = '/content/drive/MyDrive/pv-cold-start'
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT, PY, DEVICE = Path(DRIVE_REPO), sys.executable, 'cuda'
    subprocess.run([PY, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements-fm.txt')], check=True)
else:
    ROOT = Path.cwd().resolve(); ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
    PY = str(ROOT / '.venv_fm' / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')); DEVICE = 'cpu'
INP, RUNS = ROOT / 'results' / 'nb5b_inputs', ROOT / 'results' / 'nb5b_runs'
man = json.loads((INP / 'inputs_manifest.json').read_text())
for f, h in man['sha256'].items():
    assert hashlib.sha256((INP / f).read_bytes()).hexdigest() == h, f'{f} differs from its manifest'
GRID = [('chronos2', 336, 'full'), ('chronos2', 336, 'none'), ('chronos2', 1344, 'full'), ('chronos2', 1344, 'none'), ('timesfm25', 336, 'full'), ('timesfm25', 336, 'none'), ('timesfm25', 1344, 'full'), ('timesfm25', 1344, 'none')]
print('ROOT', ROOT, '| device', DEVICE, '| settings', len(GRID))

In [ ]:
log = []
for model, L, cov in GRID:
    out = RUNS / f'{model}_c{L}_{cov}.parquet'
    js = out.with_suffix('.json')
    if out.exists() and js.exists() and json.loads(js.read_text())['inputs_sha256'] == man['sha256']:
        log.append({'setting': out.stem, 'status': 'done before', **{k: json.loads(js.read_text())[k] for k in ('runtime_s', 'n_rows')}})
        continue
    t1 = time.time()
    r = subprocess.run([PY, str(ROOT / 'tools' / 'fm_infer.py'), '--model', model, '--context', str(L), '--cov', cov,
                        '--inputs', str(INP), '--out', str(out), '--device', DEVICE], cwd=str(ROOT))
    assert r.returncode == 0, f'{out.stem} failed'
    log.append({'setting': out.stem, 'status': 'run', 'runtime_s': round(time.time() - t1, 1),
                'n_rows': json.loads(js.read_text())['n_rows']})
    print(log[-1], flush=True)
print(log)

In [ ]:
if ON_COLAB:
    drive.flush_and_unmount()